# Melodic EDM Core V2 — Colab Pro inference

[Open the current V2 notebook in Colab](https://colab.research.google.com/github/Bangchis/melodic-edm-training-pipeline/blob/agent/training-v2-r32/notebooks/melodic_edm_core_v2_colab.ipynb)

This notebook installs the pinned ACE-Step source, downloads pinned XL-Base weights and the private **R32 experimental preview**, verifies checksums, generates audio, and validates 48 kHz stereo output. It performs inference only. The preview passed 6/15 multi-seed samples (40%): useful for hands-on testing, but not the final retrained release.

## 1. Before running

Sign in to the Google account that owns Colab Pro in this browser, then select **Runtime → Change runtime type → NVIDIA GPU**. In Colab Secrets (key icon), add a Hugging Face read token named `HF_TOKEN`. Add `OPENROUTER_API_KEY` only when `USE_OPENROUTER_ENHANCER = True`; enable notebook access for the secrets you use. Do not paste a token into a cell. No Google password, cookie or OAuth token belongs on the Vast server. Consumer Colab Pro has no supported server-side job-submission CLI; this notebook runs in the browser-created Colab runtime.

In [ ]:
import shutil, subprocess, torch
assert torch.cuda.is_available(), 'No NVIDIA GPU. Change the Colab runtime type.'
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
disk = shutil.disk_usage('/content')
print(f'GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB')
print(f'Disk free: {disk.free / 1024**3:.1f} GiB')
assert disk.free / 1024**3 >= 35, 'At least 35 GiB free disk is required.'
OFFLOAD_TO_CPU = vram_gib < 20
print('CPU offload:', OFFLOAD_TO_CPU)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!python -m pip -q install 'huggingface_hub>=0.30' uv

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'HF_TOKEN is missing from Colab Secrets.'
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY') or ''
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['MPLBACKEND'] = 'Agg'  # prevent Colab's notebook-only backend leaking into ACE-Step
print('Hugging Face key loaded. OpenRouter key present:', bool(OPENROUTER_API_KEY))

## 2. All generation controls (edit only this cell)

Every prompt, adapter, ACE Thinking, sampler and output setting that can change the result is collected below. `DURATION_SECONDS` and `SECTIONS` are independent editable controls: 180 seconds and seven sections are only a starter preset, and the notebook never auto-forces either value. Section labels may be descriptive custom names such as `Atmospheric Intro` or `First Melodic Drop`; only empty, duplicate, bracketed or multiline labels are rejected because they break ACE structure syntax. A short duration with many sections may collapse structure, so the notebook prints a warning instead of blocking the run. `CUSTOM_LYRICS` can override the generated section text completely before section validation. Begin with a concrete 40–80 word caption. More keywords or a longer caption are not automatically better, and conflicting required terms can reduce prompt adherence. `CUSTOM_TIMESTEPS` overrides both `INFERENCE_STEPS` and `SHIFT`. Post-processing can change level/tone but cannot repair a collapsed melody.

In [ ]:
# Model/prompt route
ADAPTER_CHOICE = 'experimental-r32'  # this temporary preview; future repos may also contain best-val/final-all-data
USE_LORA = True
LORA_SCALE = 0.5          # best historical balance; compare 0.0/base, 0.25, 0.5 and 1.0 with the same seed
USE_OPENROUTER_ENHANCER = True
OPENROUTER_MODEL = '~google/gemini-flash-latest'
USER_IDEA = ('Nhạc EDM Trung Hoa không lời, phiêu lưu và tươi sáng, có hook pipa dễ nhớ, '
             'dizi đối đáp, mở đầu không khí rồi build ngắn và drop mạnh.')
REQUIRED_PROMPT_TERMS = ['pipa', 'dizi']  # keep only truly mandatory audible terms; 0–3 is safer than a long list
DIRECT_CAPTION = ('Instrumental Chinese melodic gaming EDM with an uplifting adventurous mood, '
                  'a memorable pentatonic pipa hook, airy dizi responses, a compact build, wide '
                  'supersaw chords, clean sub bass and a punchy four-on-the-floor melodic drop.')
BPM = 128
KEYSCALE = 'F# minor'
TIME_SIGNATURE = '4'
SECTIONS = ['Intro', 'Theme', 'Build', 'Drop', 'Break', 'Final Drop', 'Outro']  # descriptive custom labels are allowed
CUSTOM_LYRICS = ''        # optional exact ACE section/lyrics text; non-empty overrides SECTIONS
DURATION_SECONDS = 180    # freely editable and never changed from the section count
WARN_SECTION_SECONDS_BELOW = 15.0  # warning only; set None to disable it

# XL-Base sampling controls
INFERENCE_STEPS = 64       # validated clean XL-Base default; still fully adjustable
GUIDANCE_SCALE = 8.0       # validated prompt-guidance default
SHIFT = 1.0                # XL-Base reference; ignored when CUSTOM_TIMESTEPS is set
USE_ADG = True             # validated for cleaner XL-Base generation
CFG_INTERVAL_START = 0.0
CFG_INTERVAL_END = 1.0
INFER_METHOD = 'ode'       # 'ode' or 'sde'
SAMPLER_MODE = 'euler'     # 'euler' or 'heun'
VELOCITY_NORM_THRESHOLD = 0.0
VELOCITY_EMA_FACTOR = 0.0
CUSTOM_TIMESTEPS = None    # e.g. [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]

# Optional ACE 5 Hz LM planner (inference only; it was not trained with the LoRA)
USE_ACE_LM_THINKING = False  # keep False for direct Base/LoRA A/B; enable separately to test planning
ACE_LM_MODEL = 'acestep-5Hz-lm-1.7B'
LM_TEMPERATURE = 0.8
LM_CFG_SCALE = 2.0
LM_TOP_K = 0
LM_TOP_P = 0.9
USE_COT_METAS = False
USE_COT_CAPTION = False
USE_COT_LANGUAGE = False
USE_COT_LYRICS = False

# Differential Correction in Wavelet domain
DCW_ENABLED = False        # DCW caused severe artifacts in the fixed-seed XL-Base A/B
DCW_MODE = 'double'        # 'low', 'high', 'double', or 'pix'
DCW_SCALER = 0.05
DCW_HIGH_SCALER = 0.02
DCW_WAVELET = 'haar'

# Decode/post-processing and output controls
ENABLE_NORMALIZATION = True
NORMALIZATION_DB = -1.0
FADE_IN_SECONDS = 0.0
FADE_OUT_SECONDS = 0.0
LATENT_SHIFT = 0.0
LATENT_RESCALE = 1.0
BATCH_SIZE = 1
USE_RANDOM_SEED = False
SEEDS = [260718]           # exactly BATCH_SIZE seeds when random seed is False
AUDIO_FORMAT = 'wav'       # mp3, wav, flac, wav32, opus, or aac
MP3_BITRATE = '320k'
MP3_SAMPLE_RATE = 48000
OUTPUT_DIR = '/content/generated-v2'

## 3. Pin every upstream component

This preview is downloaded from the private `Bangchis/melodic-edm-core-v2-r32-experimental` repository. The notebook resolves its current head to one immutable commit SHA, prints that SHA, and uses only that SHA for the whole run. Save the printed SHA with any result you want to reproduce.

In [ ]:
ACE_SOURCE_REVISION = '6d467e4b5081ccb0abf1ec1bf4fdf9051a2d34b0'
ACE_CORE_REVISION = '19671f406d603126926c1b7e2adc169acbcade22'
XL_BASE_REVISION = '220c1166efbdd9583eafcb12eb160594bbfcb241'
RELEASE_REPO = 'Bangchis/melodic-edm-core-v2-r32-experimental'
from huggingface_hub import HfApi
RELEASE_REVISION = HfApi(token=HF_TOKEN).model_info(RELEASE_REPO).sha
assert RELEASE_REVISION and len(RELEASE_REVISION) == 40, 'Hugging Face did not return an immutable release SHA.'
print('Pinned V2 release revision:', RELEASE_REVISION)

In [ ]:
from pathlib import Path
ACE_ROOT = Path('/content/ACE-Step-1.5')
if not ACE_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ace-step/ACE-Step-1.5.git', str(ACE_ROOT)], check=True)
subprocess.run(['git', '-C', str(ACE_ROOT), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(ACE_ROOT), 'checkout', '--detach', ACE_SOURCE_REVISION], check=True)
subprocess.run(['uv', 'sync'], cwd=ACE_ROOT, check=True)
print('ACE-Step source and environment ready at', ACE_SOURCE_REVISION)

In [ ]:
from huggingface_hub import snapshot_download
CHECKPOINT_ROOT = ACE_ROOT / 'checkpoints'
snapshot_download(
    repo_id='ACE-Step/Ace-Step1.5', revision=ACE_CORE_REVISION,
    local_dir=CHECKPOINT_ROOT, token=HF_TOKEN,
)
snapshot_download(
    repo_id='ACE-Step/acestep-v15-xl-base', revision=XL_BASE_REVISION,
    local_dir=CHECKPOINT_ROOT / 'acestep-v15-xl-base', token=HF_TOKEN,
)
print('Pinned ACE-Step core and XL-Base checkpoints downloaded.')

In [ ]:
RELEASE_DIR = Path('/content/melodic-edm-core-v2-r32-experimental')
snapshot_download(
    repo_id=RELEASE_REPO, revision=RELEASE_REVISION,
    repo_type='model', local_dir=RELEASE_DIR, token=HF_TOKEN,
)
available_adapters = [name for name in ('experimental-r32', 'final-all-data', 'best-val') if (RELEASE_DIR / name / 'adapter_config.json').is_file()]
if ADAPTER_CHOICE == 'auto':
    if not available_adapters:
        raise FileNotFoundError('No packaged adapter exists in this release.')
    ADAPTER = available_adapters[0]
elif ADAPTER_CHOICE in ('experimental-r32', 'final-all-data', 'best-val'):
    ADAPTER = ADAPTER_CHOICE
else:
    raise ValueError("ADAPTER_CHOICE must be 'auto', 'experimental-r32', 'final-all-data', or 'best-val'")
assert (RELEASE_DIR / ADAPTER / 'adapter_config.json').is_file()
assert (RELEASE_DIR / ADAPTER / 'adapter_model.safetensors').is_file()
subprocess.run(['sha256sum', '-c', 'SHA256SUMS'], cwd=RELEASE_DIR, check=True)
print('Private V2 release downloaded and every packaged checksum passed. Adapter:', ADAPTER)

## 4. Compile the selected prompt route

When the enhancer switch is on, the notebook sends only your idea and explicit musical conditions to OpenRouter, validates exactly five music fields, and compiles a 40–300 word caption. When it is off, `DIRECT_CAPTION` goes straight to ACE-Step and no OpenRouter key or request is used.

In [ ]:
import importlib, json, sys
sys.path.insert(0, str(RELEASE_DIR / 'scripts'))
import enhance_prompt_openrouter as prompt_enhancer_module
prompt_enhancer_module = importlib.reload(prompt_enhancer_module)  # do not reuse a stale pre-hotfix Colab import
enhance_prompt = prompt_enhancer_module.enhance_prompt
sections_to_lyrics = prompt_enhancer_module.sections_to_lyrics
try:
    sections_to_lyrics(['Atmospheric Intro', 'First Melodic Drop'])
except ValueError as error:
    raise RuntimeError(
        'A stale release is loaded and still restricts section names. Restart the Colab runtime, '
        'open the latest public notebook, and run every cell from the top.'
    ) from error
STRUCTURE_LYRICS = CUSTOM_LYRICS.strip() or sections_to_lyrics(SECTIONS)
EXPLICIT_CONDITIONS = {
    'bpm': BPM, 'keyscale': KEYSCALE, 'timesignature': TIME_SIGNATURE,
    'required_terms': REQUIRED_PROMPT_TERMS,
}
if not CUSTOM_LYRICS.strip():
    EXPLICIT_CONDITIONS['sections'] = SECTIONS
if USE_OPENROUTER_ENHANCER:
    assert OPENROUTER_API_KEY, 'Add OPENROUTER_API_KEY to Colab Secrets or disable the enhancer.'
    enhancement = enhance_prompt(
        USER_IDEA, OPENROUTER_API_KEY, model=OPENROUTER_MODEL, explicit_conditions=EXPLICIT_CONDITIONS,
    )
    music_conditions = enhancement['conditions']
    print('OpenRouter resolved model:', enhancement['resolved_model'])
else:
    assert DIRECT_CAPTION.strip(), 'DIRECT_CAPTION cannot be empty when the enhancer is disabled.'
    music_conditions = {
        'caption': DIRECT_CAPTION.strip(), 'bpm': BPM, 'keyscale': KEYSCALE,
        'timesignature': TIME_SIGNATURE, 'lyrics': STRUCTURE_LYRICS,
    }
    enhancement = {'enabled': False, 'conditions': music_conditions}
    print('OpenRouter enhancer disabled: using DIRECT_CAPTION unchanged.')
music_conditions['lyrics'] = STRUCTURE_LYRICS
if not CUSTOM_LYRICS.strip() and SECTIONS and WARN_SECTION_SECONDS_BELOW is not None:
    seconds_per_section = DURATION_SECONDS / len(SECTIONS)
    if seconds_per_section < WARN_SECTION_SECONDS_BELOW:
        print(f'WARNING: only {seconds_per_section:.1f}s per section; this may compress or collapse structure. Values are unchanged.')
print('Caption:', music_conditions['caption'])
print('Duration/sections are user-controlled:', DURATION_SECONDS, 'seconds /', len(SECTIONS), 'sections')
sampling_settings = {
    'inference_steps': INFERENCE_STEPS, 'guidance_scale': GUIDANCE_SCALE, 'shift': SHIFT,
    'use_adg': USE_ADG, 'cfg_interval_start': CFG_INTERVAL_START,
    'cfg_interval_end': CFG_INTERVAL_END, 'infer_method': INFER_METHOD,
    'sampler_mode': SAMPLER_MODE, 'velocity_norm_threshold': VELOCITY_NORM_THRESHOLD,
    'velocity_ema_factor': VELOCITY_EMA_FACTOR, 'dcw_enabled': DCW_ENABLED,
    'dcw_mode': DCW_MODE, 'dcw_scaler': DCW_SCALER,
    'dcw_high_scaler': DCW_HIGH_SCALER, 'dcw_wavelet': DCW_WAVELET,
    'timesteps': CUSTOM_TIMESTEPS, 'enable_normalization': ENABLE_NORMALIZATION,
    'normalization_db': NORMALIZATION_DB, 'fade_in_duration': FADE_IN_SECONDS,
    'fade_out_duration': FADE_OUT_SECONDS, 'latent_shift': LATENT_SHIFT,
    'latent_rescale': LATENT_RESCALE, 'thinking': USE_ACE_LM_THINKING,
    'ace_lm_model': ACE_LM_MODEL, 'lm_temperature': LM_TEMPERATURE,
    'lm_cfg_scale': LM_CFG_SCALE, 'lm_top_k': LM_TOP_K, 'lm_top_p': LM_TOP_P,
    'use_cot_metas': USE_COT_METAS, 'use_cot_caption': USE_COT_CAPTION,
    'use_cot_language': USE_COT_LANGUAGE, 'use_cot_lyrics': USE_COT_LYRICS,
}
output_settings = {
    'batch_size': BATCH_SIZE, 'use_random_seed': USE_RANDOM_SEED,
    'seeds': None if USE_RANDOM_SEED else SEEDS, 'audio_format': AUDIO_FORMAT,
    'mp3_bitrate': MP3_BITRATE, 'mp3_sample_rate': MP3_SAMPLE_RATE,
}
custom_prompt = {
    'sampling': sampling_settings, 'output': output_settings,
    'prompts': [{
        'id': 'colab_custom_01',
        'caption': music_conditions['caption'],
        'bpm': music_conditions['bpm'],
        'keyscale': music_conditions['keyscale'],
        'timesignature': music_conditions['timesignature'],
        'lyrics': music_conditions['lyrics'],
        'duration': DURATION_SECONDS, 'seed': SEEDS[0] if SEEDS else -1,
    }]
}
ENHANCEMENT_FILE = Path('/content/v2_prompt_enhancement.json')
ENHANCEMENT_FILE.write_text(json.dumps(enhancement, ensure_ascii=False, indent=2), encoding='utf-8')
PROMPT_FILE = Path('/content/v2_custom_prompt.json')
PROMPT_FILE.write_text(json.dumps(custom_prompt, ensure_ascii=False, indent=2), encoding='utf-8')
print('Complete inference payload ready:', json.dumps(custom_prompt, ensure_ascii=False, indent=2))

In [ ]:
OUTPUT_DIR = Path(OUTPUT_DIR)
command = [
    str(ACE_ROOT / '.venv/bin/python'),
    str(RELEASE_DIR / 'scripts/infer_v2_release.py'),
    '--ace-root', str(ACE_ROOT),
    '--checkpoint-root', str(CHECKPOINT_ROOT),
    '--release-dir', str(RELEASE_DIR),
    '--adapter-subdirectory', ADAPTER,
    '--prompts', str(PROMPT_FILE),
    '--prompt-index', '0',
    '--output-dir', str(OUTPUT_DIR),
]
if not USE_LORA or LORA_SCALE == 0:
    command.append('--disable-lora')
else:
    command.extend(['--lora-scale', str(LORA_SCALE)])
if OFFLOAD_TO_CPU:
    command.append('--offload-to-cpu')
print('Starting ACE-Step inference. Full output is also saved to /content/v2_inference.log')
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ, 'HF_TOKEN': HF_TOKEN, 'MPLBACKEND': 'Agg'},
)
inference_log_lines = []
for line in process.stdout:
    print(line, end='')
    inference_log_lines.append(line)
return_code = process.wait()
INFERENCE_LOG = Path('/content/v2_inference.log')
INFERENCE_LOG.write_text(''.join(inference_log_lines), encoding='utf-8')
if return_code != 0:
    tail = ''.join(inference_log_lines[-120:])
    raise RuntimeError(
        f'ACE-Step inference exited with code {return_code}. Full log: {INFERENCE_LOG}\n'
        f'Last output lines:\n{tail}'
    )

In [ ]:
from IPython.display import Audio, display
report = json.loads((OUTPUT_DIR / 'inference_report.json').read_text(encoding='utf-8'))
assert report['status'] == 'pass'
assert report['probe']['sample_rate'] == 48000
assert report['probe']['channels'] == 2
assert report['probe']['duration'] >= 10
print(json.dumps(report, indent=2))
for item in report['audios']:
    display(Audio(item['audio_path']))

## Troubleshooting

- **CUDA OOM:** restart the runtime, set `OFFLOAD_TO_CPU = True`, reduce `BATCH_SIZE`, duration or sampling steps.
- **Hugging Face 401/403:** confirm `HF_TOKEN` can read the private model and Secret access is enabled.
- **OpenRouter 401/402/429:** verify `OPENROUTER_API_KEY`, credits and rate limit, or set `USE_OPENROUTER_ENHANCER = False`.
- **Checksum failure:** delete `/content/melodic-edm-core-v2-r32-experimental` and download the same immutable revision again.
- **Missing adapter:** keep `experimental-r32` for this preview.
- **Chaotic/looped audio:** restore the preset, keep Thinking off, use 40–80 words, allow about 20–30 seconds per section, and compare Base versus LoRA on the same seed. Do not try to fix collapse with normalization.
- **Initialization failure:** verify both pinned checkpoint downloads completed. Do not substitute another base model.
- **Inference subprocess failure:** open `/content/v2_inference.log` or inspect the final 120 lines printed by the generation cell; `CalledProcessError` alone is only a wrapper, not the underlying error.